# Semantic search

In [3]:
import sys
print(sys.executable)


c:\Users\meisa\Desktop\Code\ML\NLP\HuggingFace_course\.env\Scripts\python.exe


In [ ]:
# Installing the necessary libraries
!{sys.executable} -m pip install sentence-transformers==2.2.0
!{sys.executable} -m pip install datasets==2.0.0
#!pip install torch==1.12.0

In [ ]:
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install datasets

In [1]:
# Importing necessary modules
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
import torch
import os


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\meisa\Desktop\Code\ML\NLP\HuggingFace_course\.env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance(

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
# Load a sample from the Multi-News dataset and convert it to a pandas dataframe
dataset = load_dataset("multi_news", split="test")
df = dataset.to_pandas().sample(2000, random_state=42)

In [ ]:
# Load the SentenceTransformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode the summaries and store them as embeddings
df["embedding"] = list(model.encode(df["summary"].to_list(), show_progress_bar=True))
passage_embeddings = df["embedding"].to_list()

In [ ]:
def find_relevant_news(query):
    # Encode the query using the same model
    query_embedding = model.encode(query)

    # Calculate the cosine similarity between the query and passage embeddings
    similarities = util.cos_sim(query_embedding, passage_embeddings)

    # Get the indices of the top 3 most similar passages
    top_indices = torch.topk(similarities.flatten(), 3).indices

    # Retrieve the summaries of the top 3 passages and truncate them to 160 characters
    top_relevant_passages = [df.iloc[x.item()]["summary"][:160] + "..." for x in top_indices]

    return top_relevant_passages

In [ ]:
def clear_screen():
    os.system("clear")

In [ ]:
def interactive_search():
    print("Welcome to the Semantic News Search!\n")
    while True:
        print("Type in a topic you'd like to find articles about, and I'll do the searching! (Type 'exit' to quit)\n> ", end="")

        query = input().strip()

        if query.lower() == "exit":
            print("\nThanks for using the Semantic News Search! Have a great day!")
            break

        print("\n\tHere are 3 articles I found based on your query: \n")

        passages = find_relevant_news(query)
        for passage in passages:
            print("\n\t" + passage)

        input("\nPress Enter to continue searching...")
        clear_screen()

In [ ]:
# Start the interactive search
interactive_search()